**Data cleaning table wise**

In [2]:
import pandas as pd
import numpy as np

In [ ]:
# Load normal CSV files
customers = pd.read_csv("raw_data/olist_customers_dataset.csv")
sellers = pd.read_csv("raw_data/olist_sellers_dataset.csv")
category_translation = pd.read_csv("raw_data/product_category_name_translation.csv")

# Load zipped CSV files
geolocation = pd.read_csv("raw_data/olist_geolocation_dataset.csv.zip", compression="zip")
order_items = pd.read_csv("raw_data/olist_order_items_dataset.csv.zip", compression="zip")
order_payments = pd.read_csv("raw_data/olist_order_payments_dataset.csv.zip", compression="zip")
order_reviews = pd.read_csv("raw_data/olist_order_reviews_dataset.csv.zip", compression="zip")
orders = pd.read_csv("raw_data/olist_orders_dataset.csv.zip", compression="zip")
products = pd.read_csv("raw_data/olist_products_dataset.csv.zip", compression="zip")



In [4]:
#customers 
# Clean city names: remove whitespace and capitalize
customers ['customer_city'] = customers ['customer_city'].str.strip().str.capitalize()

# Standardize state abbreviations to uppercase
customers ['customer_state'] = customers ['customer_state'].str.upper()

# Convert to string and fill leading zeros to make 5 digits
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)

In [5]:
customers.head()


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP


In [6]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [7]:
customers['customer_zip_code_prefix'].str.len()

0        5
1        5
2        5
3        5
4        5
        ..
99436    5
99437    5
99438    5
99439    5
99440    5
Name: customer_zip_code_prefix, Length: 99441, dtype: int64

In [8]:
#geolocation

import unidecode

# Clean city names: remove whitespace and capitalize
geolocation['geolocation_city'] = geolocation['geolocation_city'].str.strip().str.capitalize()

# Standardize state abbreviations to uppercase
geolocation['geolocation_state'] = geolocation['geolocation_state'].str.upper()



In [9]:
# converting to English version of city names without special characters
geolocation['geolocation_city'] = geolocation['geolocation_city'].apply(lambda x: unidecode.unidecode(x))

In [10]:
# Valid latitude range for Brazil
geolocation = geolocation[(geolocation['geolocation_lat'] >= -33) & (geolocation['geolocation_lat'] <= 5)]

# Valid longitude range for Brazil
geolocation = geolocation[(geolocation['geolocation_lng'] >= -74) & (geolocation['geolocation_lng'] <= -34)]

In [11]:
geolocation.head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,Sao paulo,SP
1,1046,-23.546081,-46.644820,Sao paulo,SP
2,1046,-23.546129,-46.642951,Sao paulo,SP
3,1041,-23.544392,-46.639499,Sao paulo,SP
4,1035,-23.541578,-46.641607,Sao paulo,SP
5,1012,-23.547762,-46.635361,Sao paulo,SP
6,1047,-23.546273,-46.641225,Sao paulo,SP
7,1013,-23.546923,-46.634264,Sao paulo,SP
8,1029,-23.543769,-46.634278,Sao paulo,SP
9,1011,-23.547640,-46.636032,Sao paulo,SP


In [12]:
geolocation.duplicated().sum()

279602

In [13]:
#order_item table

# Converting shipping limit date to datetime

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

In [14]:
# Calculate total value per item (price + freight)
order_items['total_value'] = order_items['price'] + order_items['freight_value']

In [15]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04


In [16]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
 7   total_value          112650 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 6.9 MB


In [17]:
#order_payments table

order_payments['payment_type'] = order_payments['payment_type'].str.strip().str.lower()

In [18]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [19]:
#Checking outliners

order_payments[order_payments['payment_value'] == 13664.08]
order_items[order_items['order_id'] == '03caa2c082116e1d31e67e9ae3700499']
#Probably a bulk order, not an outlinear

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_value
1647,03caa2c082116e1d31e67e9ae3700499,1,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1648,03caa2c082116e1d31e67e9ae3700499,2,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1649,03caa2c082116e1d31e67e9ae3700499,3,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1650,03caa2c082116e1d31e67e9ae3700499,4,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1651,03caa2c082116e1d31e67e9ae3700499,5,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1652,03caa2c082116e1d31e67e9ae3700499,6,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1653,03caa2c082116e1d31e67e9ae3700499,7,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01
1654,03caa2c082116e1d31e67e9ae3700499,8,5769ef0a239114ac3a854af00df129e4,b37c4c02bda3161a7546a4e6d222d5b2,2017-10-06 15:28:20,1680.0,28.01,1708.01


In [20]:
df = order_payments[order_payments['payment_value']==0]
print(df)
df.shape

                                order_id  payment_sequential payment_type  \
19922   8bcbe01d44d147f901cd3192671144db                   4      voucher   
36822   fa65dad1b0e818e3ccc5cb0e39231352                  14      voucher   
43744   6ccb433e00daae1283ccc956189c82ae                   4      voucher   
51280   4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411   00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
62674   45ed6e85398a87c253db47c2d9f48216                   3      voucher   
77885   fa65dad1b0e818e3ccc5cb0e39231352                  13      voucher   
94427   c8c528189310eaa44a745b8d9d26908b                   1  not_defined   
100766  b23878b3e8eb4d25a158f57d96331b18                   4      voucher   

        payment_installments  payment_value  
19922                      1            0.0  
36822                      1            0.0  
43744                      1            0.0  
51280                      1            0.0  

(9, 5)

In [21]:
zero_orders = order_payments[order_payments['payment_value'] == 0]

zero_orders = zero_orders.merge(
    orders[['order_id','order_status']],
    on='order_id',
    how='left')

zero_orders
#May not ne a data error, not removing beacuse of further analysiss

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,shipped
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,delivered
3,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled
4,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled
5,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0,delivered
6,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0,shipped
7,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled
8,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0,delivered


In [22]:
#order_reviews table

# chamging date datatype
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'], errors='coerce')
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'], errors='coerce')

In [23]:
order_reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [24]:
#Filling null values
order_reviews['review_comment_title'] = \
      order_reviews['review_comment_title'].fillna('No Title')

order_reviews['review_comment_message'] = \
      order_reviews['review_comment_message'].fillna('No Comments')

In [25]:
#Creating review category based on review score

def review_category(score):
    if score <= 3:
        return "Bad"
    elif score > 3 and score <= 4:
        return "Good"
    elif score > 4:
        return "Very Good"
    else:
        return "No Review"

order_reviews['review_category'] = order_reviews['review_score'].apply(review_category)

In [26]:
#Calculating feedback time
order_reviews['feedback_timestamp'] = order_reviews['review_answer_timestamp'] - order_reviews['review_creation_date']

In [27]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_category,feedback_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Title,No Comments,2018-01-18,2018-01-18 21:46:59,Good,0 days 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Title,No Comments,2018-03-10,2018-03-11 03:05:13,Very Good,1 days 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Title,No Comments,2018-02-17,2018-02-18 14:36:24,Very Good,1 days 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,No Title,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,Very Good,0 days 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,No Title,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,Very Good,1 days 10:26:53


In [28]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype          
---  ------                   --------------  -----          
 0   review_id                99224 non-null  str            
 1   order_id                 99224 non-null  str            
 2   review_score             99224 non-null  int64          
 3   review_comment_title     99224 non-null  str            
 4   review_comment_message   99224 non-null  str            
 5   review_creation_date     99224 non-null  datetime64[us] 
 6   review_answer_timestamp  99224 non-null  datetime64[us] 
 7   review_category          99224 non-null  str            
 8   feedback_timestamp       99224 non-null  timedelta64[us]
dtypes: datetime64[us](2), int64(1), str(5), timedelta64[us](1)
memory usage: 6.8 MB


In [29]:
#orders table

#Changing date datatype

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce')
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'], errors='coerce')
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'], errors='coerce')
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'], errors='coerce')
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'], errors='coerce')

In [30]:
# calculating delivery time
orders['delivery_time'] = orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
orders['order_estimated_delivery_time'] = orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']
orders['order_shipping_time'] = orders['order_delivered_carrier_date'] - orders['order_approved_at']
orders['order_processing_time'] = orders['order_approved_at'] - orders['order_purchase_timestamp']
orders['order_transit_time'] = orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']

In [31]:
# Delivered orders
orders['is_delivered'] = orders['order_delivered_customer_date'].notna()

# Canceled orders
orders['is_canceled'] = orders['order_status'] == 'canceled'

In [32]:
# standerdize lowercase and remove whitespace

orders['order_status'] = orders['order_status'].str.lower().str.strip()

In [33]:
orders.isnull().sum()
#Not removing null values because of further analysis, may be useful for understanding delivery times and delays

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_time                    2965
order_estimated_delivery_time       0
order_shipping_time              1797
order_processing_time             160
order_transit_time               2966
is_delivered                        0
is_canceled                         0
dtype: int64

In [34]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time,order_estimated_delivery_time,order_shipping_time,order_processing_time,order_transit_time,is_delivered,is_canceled
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8 days 10:28:40,15 days 13:03:27,2 days 08:47:45,0 days 00:10:42,6 days 01:30:13,True,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13 days 18:46:08,19 days 03:18:23,0 days 11:06:33,1 days 06:42:50,12 days 00:56:45,True,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9 days 09:27:40,26 days 15:21:11,0 days 04:54:37,0 days 00:16:34,9 days 04:16:29,True,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13 days 05:00:36,26 days 04:31:54,3 days 17:54:00,0 days 00:17:53,9 days 10:48:43,True,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2 days 20:58:23,12 days 02:41:21,0 days 21:26:05,0 days 01:01:50,1 days 22:30:28,True,False


In [35]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype          
---  ------                         --------------  -----          
 0   order_id                       99441 non-null  str            
 1   customer_id                    99441 non-null  str            
 2   order_status                   99441 non-null  str            
 3   order_purchase_timestamp       99441 non-null  datetime64[us] 
 4   order_approved_at              99281 non-null  datetime64[us] 
 5   order_delivered_carrier_date   97658 non-null  datetime64[us] 
 6   order_delivered_customer_date  96476 non-null  datetime64[us] 
 7   order_estimated_delivery_date  99441 non-null  datetime64[us] 
 8   delivery_time                  96476 non-null  timedelta64[us]
 9   order_estimated_delivery_time  99441 non-null  timedelta64[us]
 10  order_shipping_time            97644 non-null  timedelta64[us]
 11  order_process

In [36]:
#products table

# Fill missing values instead of dropping

products['product_category_name'] = \
    products['product_category_name'].fillna('unknown').str.strip().str.lower()

products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)


In [37]:
# weight category

def weight_category(weight):
    if pd.isnull(weight):  # Handle missing weights
        return 'unknown'
    elif weight <= 300:
        return 'light'
    elif 300 < weight <= 1000:
        return 'medium'
    elif weight > 1000:
        return 'heavy'


products['weight_category'] = products['product_weight_g'].apply(weight_category)


In [38]:
# Checking large products by volume

def size_category(length, width, height):
    # Check for missing or invalid dimensions
    if pd.isnull(length) or pd.isnull(width) or pd.isnull(height):
        return 'unknown'
    if length <= 0 or width <= 0 or height <= 0:
        return 'unknown'
    
    # Calculate volume
    volume = length * width * height
    
    # Categorize based on volume
    if volume <= 1000:
        return 'small'
    elif 1000 < volume <= 10000:
        return 'medium'
    else:
        return 'large'

# Apply the function to create a new column
products['size_category'] = products.apply(
    lambda row: size_category(row['product_length_cm'],
                              row['product_width_cm'],
                              row['product_height_cm']),
    axis=1
)

In [39]:
# Merge on 'product_category_name'
products_cleaned = products.merge(
category_translation[['product_category_name','product_category_name_english']],
    on='product_category_name',
    how='left')

In [40]:
products_cleaned.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,weight_category,size_category,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,light,medium,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,medium,large,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,light,medium,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,medium,medium,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,medium,medium,housewares


In [41]:
products.head()


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,weight_category,size_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,light,medium
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,medium,large
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,light,medium
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,medium,medium
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,medium,medium


In [42]:
products.isnull().sum()
#Not removing null values of numerical columns because of further analysis, may be useful for understanding product categories and their impact on sales and delivery times

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              2
product_length_cm             2
product_height_cm             2
product_width_cm              2
weight_category               0
size_category                 0
dtype: int64

In [43]:
# standerdize lowercase and remove whitespace

products['product_category_name'] = products['product_category_name'].str.strip().str.lower()

In [44]:
products[products['product_weight_g'] == 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,weight_category,size_category
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0,light,large
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0,light,large
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0,light,large
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0,light,large


In [45]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32951 non-null  str    
 2   product_name_lenght         32951 non-null  float64
 3   product_description_lenght  32951 non-null  float64
 4   product_photos_qty          32951 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
 9   weight_category             32951 non-null  str    
 10  size_category               32951 non-null  str    
dtypes: float64(7), str(4)
memory usage: 2.8 MB


In [46]:
#sellers table
 
# Clean city names: remove whitespace and capitalize
sellers['seller_city'] = sellers['seller_city'].str.strip().str.capitalize()

# Standardize state abbreviations to uppercase
sellers['seller_state'] = sellers['seller_state'].str.upper()

# Convert to string and fill leading zeros to make 5 digits
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

In [47]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,Sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca paulista,SP


In [48]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 96.8 KB


In [49]:
#category translation table

# standerdize category names to lowercase and remove whitespace

category_translation['product_category_name'] = category_translation['product_category_name'].str.strip().str.lower()
category_translation['product_category_name_english'] = category_translation['product_category_name_english'].str.strip().str.lower()

In [50]:
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [ ]:
# saving to csv files
# Customers
customers.to_csv('customers_cleaned.csv', index=False)

# Orders
orders.to_csv('orders_cleaned.csv', index=False)

# Products
products.to_csv('products_cleaned.csv', index=False)

# Order Reviews
order_reviews.to_csv('order_reviews_cleaned.csv', index=False)

# Order Payments
order_payments.to_csv('order_payments_cleaned.csv', index=False)

# Order Items
order_items.to_csv('order_items_cleaned.csv', index=False)

# Geolocation
geolocation.to_csv('geolocation_cleaned.csv', index=False)

# Category Translation
category_translation.to_csv('category_translation_cleaned.csv', index=False)

# Sellers
sellers.to_csv('sellers_cleaned.csv', index=False)


# Cleaned Products (your merged final product dataset)
products_cleaned.to_csv('cleaned_products_final.csv', index=False)

print("All CSV files saved successfully!")

All CSV files saved successfully!
